In [ ]:
"""
Post-processing of the GridMET climatology data
~ Build ERC inputs for FSPro by CO Pyrome
"""

import os, sys

from pathlib import Path
from fb_tools.weather import (
    load_gridmet_csv,
    build_historic_erc_arrays,
    build_erc_stats,
    build_erc_classes,
    build_current_erc_values,
)

# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

## Post-processing: Build per-pyrome FSPro weather inputs from exported CSV

Once the GEE export completes and the CSV is downloaded to ``data/weather/``,
run the cells below to build FSPro-ready ERC arrays and ERC class tables.

**Expected CSV path:** ``data/weather/gridmet_erc_pyromes.csv``

In [ ]:
# --- Paths
csv_path = projdir / 'data' / 'tabular' / 'raw' / 'weather' / 'gridmet_clim_CO_pyromes.csv'
erc_cache_dir = projdir / 'data' / 'weather' / 'pyrome_erc'
erc_cache_dir.mkdir(parents=True, exist_ok=True)

# --- Load and inspect
clim = load_gridmet_csv(csv_path)
print(f"Loaded {len(clim):,} rows × {clim.shape[1]} columns")
print(clim.dtypes)
clim.head()

In [ ]:
# --- Clean up some of the columns
clim.drop(columns=['system:index','.geo'], inplace=True)
print(clim.columns)

In [ ]:
# --- Build historic ERC arrays (15yr × 214day) per pyrome + write JSON cache
historic = build_historic_erc_arrays(clim, out_dir=erc_cache_dir)
print(f"\nPyromes processed: {len(historic)}")
for pid, arr in list(historic.items())[:3]:
    print(f"  Pyrome {pid}: shape {arr.shape}  (years × DOYs)")

# --- ERC stats: average and std-dev per DOY across years
stats = build_erc_stats(historic)
print(f"\nERC stats computed for {len(stats)} pyromes")
print(f"  avg shape: {list(stats.values())[0]['avg'].shape}")

# --- ERC classes: 5-row table with FM values per pyrome
classes = build_erc_classes(clim)
print(f"\nERC classes built for {len(classes)} pyromes")
for pid, cls in list(classes.items())[:2]:
    print(f"\n  Pyrome {pid}:")
    print(f"  {'lower':>6} {'upper':>6} {'fm1':>5} {'fm10':>5} "
          f"{'fm100':>6} {'fm_herb':>7} {'fm_woody':>8} {'spot_d':>7} "
          f"{'spot_p':>7} {'spot2':>6}")
    for row in cls:
        print(f"  {row[0]:>6.0f} {row[1]:>6.0f} {row[2]:>5.1f} "
              f"{row[3]:>5.1f} {row[4]:>6.1f} {row[5]:>7.1f} {row[6]:>8.1f} "
              f"{row[7]:>7.0f} {row[8]:>7.2f} {row[9]:>6.0f}")